In [0]:
df_users_pinu=spark.read.csv('dbfs:/FileStore/Pinaki/users_data.csv',header=True)

In [0]:
df_users_pinu.show(50,truncate=True)

In [0]:
display(df_users_pinu)

In [0]:
df_users_pinu.count()

In [0]:
dbutils.fs.ls("dbfs:/FileStore/tables/Arijit")

In [0]:
%fs

ls dbfs:/FileStore/tables/Arijit

In [0]:
df_users = spark.read.csv('dbfs:/FileStore/tables/Arijit/complex_users.csv',header=True,inferSchema=True)
df_subscriptions = spark.read.csv('dbfs:/FileStore/tables/Arijit/complex_subscriptions.csv',header=True,inferSchema=True)
df_watch_history = spark.read.csv('dbfs:/FileStore/tables/Arijit/complex_watch_history.csv',header=True,inferSchema=True)
df_payments = spark.read.csv('dbfs:/FileStore/tables/Arijit/complex_payments.csv',header=True,inferSchema=True)

In [0]:
df_users.printSchema()

In [0]:
(df_users.count(), len(df_users.columns))

In [0]:
df_users.columns

In [0]:
display(df_users)

In [0]:
df_users.show(50,truncate=True)

In [0]:
df_users.show(50, truncate=False)

In [0]:
df_users.limit(100).toPandas()

In [0]:
# Display Last 50 Records
display(df_users.orderBy(df_users['user_id'].desc()).limit(50))

# Setup Paths

In [0]:
# Raw input paths (uploaded CSVs)
raw_base_path = "dbfs:/FileStore/tables/Arijit/"

users_path = raw_base_path + "complex_users.csv"
subs_path = raw_base_path + "complex_subscriptions.csv"
watch_path = raw_base_path + "complex_watch_history.csv"
payments_path = raw_base_path + "complex_payments.csv"

# Bronze output paths
bronze_base_path = "dbfs:/FileStore/tables/Arijit/bronze/"

bronze_users_path = bronze_base_path + "users"
bronze_subs_path = bronze_base_path + "subscriptions"
bronze_watch_path = bronze_base_path + "watch_history"
bronze_payments_path = bronze_base_path + "payments"

# Enable Delta Features

In [0]:
# spark.conf.set("spark.sql.shuffle.partitions", 4)
spark.conf.set("spark.databricks.delta.schema.automerge.enabled", "true") # It enables automatic schema evolution (auto merge) in Delta Lake.

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS arijit_bronze_db;

# Very Important Interview Definition

> Hive Metastore is a centralized metadata repository used by Spark and Hive to store information about databases, tables, schemas, partitions, and data locations, while the actual data is stored in distributed storage like HDFS, ADLS, S3, or DBFS.

# Raw Data INGESTION
## Read Raw Users Data

# Spark Read Modes — PERMISSIVE, DROPMALFORMED, FAILFAST

When reading CSV/JSON data in Spark, the **mode** option controls what Spark should do when it encounters **bad or malformed records**.

Example:

```python
spark.read.format("csv") \
    .option("mode", "PERMISSIVE") \
    .load(path)
```

---

# 1. PERMISSIVE (Default Mode)

This is the **default mode** in Spark.

### Behavior:

* Bad records are **not dropped**
* Bad records are **set to null**
* Corrupt records can be stored using `badRecordsPath`
* Job **does NOT fail**

**NOTE: If 'badRecordsPath' is specified, 'mode' is not allowed to set. mode: PermissiveMode**

### Example Bad Record:

CSV file:

```
id,name,age
1,Arijit,25
2,Rahul,abc
3,Amit,30
```

If `age` should be integer:

```
id name  age
1  Arijit 25
2  Rahul  null
3  Amit   30
```

### Use Case:

* Bronze layer ingestion
* Raw data ingestion
* When data quality is unknown

---

# 2. DROPMALFORMED

### Behavior:

* Drops rows that contain malformed records
* Only valid rows are loaded
* Bad rows are ignored
* Job does NOT fail

Example:

```
id,name,age
1,Arijit,25
2,Rahul,abc
3,Amit,30
```

Result:

```
1,Arijit,25
3,Amit,30
```

Row with Rahul is dropped.

### Use Case:

* When bad data should be ignored
* Silver layer sometimes
* Data cleaning pipelines

---

# 3. FAILFAST

### Behavior:

* If any bad record is found → Job fails immediately
* No data is loaded

FAILFAST is very important in production pipelines where data quality must be strictly enforced.

Example:
If one bad record exists → Spark throws error and stops.

### Use Case:

* Gold layer
* Strict data pipelines
* Financial / critical data
* When data quality must be perfect

---

# Summary Table

| Mode          | Bad Records                  | Job Failure | Use Case |
| ------------- | ---------------------------- | ----------- | -------- |
| PERMISSIVE    | Set null / store bad records | No          | Bronze   |
| DROPMALFORMED | Dropped                      | No          | Silver   |
| FAILFAST      | Not allowed                  | Yes         | Gold     |

---

# Recommended Usage in Medallion Architecture

| Layer  | Mode          |
| ------ | ------------- |
| Bronze | PERMISSIVE    |
| Silver | DROPMALFORMED |
| Gold   | FAILFAST      |

---

# Example Bronze Read

```python
spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("mode", "PERMISSIVE") \
    .option("badRecordsPath", "dbfs:/badrecords/") \
    .load(path)
```

---

# Important Note

**PERMISSIVE + badRecordsPath** is the most commonly used ingestion pattern in real-world data engineering pipelines.


In [0]:
users_df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .option("badRecordsPath", "dbfs:/FileStore/tables/Arijit/badrecords/users/") \
            .load(users_path)

display(users_df); print("Total Rows: ",users_df.count())


subs_df = spark.read.format("csv") \
                .options(header="true",
                        inferSchema="true",
                        ) \
                .option("badRecordsPath", "dbfs:/FileStore/tables/Arijit/badrecords/subscription/") \
                .load(subs_path)


display(subs_df); print("Total Rows: ",subs_df.count())


watch_df = spark.read.format("csv") \
                        .options(header="true", inferSchema="true",
                                 badRecodsPath="dbfs:/FileStore/tables/Arijit/badrecords/watch_history/") \
                        .load(watch_path)

display(watch_df); print("Total Rows: ",watch_df.count())

payments_df = spark.read.format("csv") \
                        .options(header="true", inferSchema="true",
                                 badRecordsPath="dbfs:/FileStore/tables/Arijit/badrecords/payments/") \
                        .load(payments_path)

display(payments_df); print("Total Rows: ",payments_df.count())

# Add Metadata Columns

In [0]:
# from pyspark.sql.functions import current_timestamp, input_file_name
# users_df = users_df.withColumn("ingest_file_name", input_file_name()) \
#                    .withColumn("ingest_time", current_timestamp())

# display(users_df)
# users_df.write.format("delta").mode("overwrite").save(bronze_users_path)
# subs_df = spark.read.format("csv") \
#             .option("header", "true") \
#             .option("inferSchema", "true") \
#             .option("badRecordsPath", "dbfs:/FileStore/tables/Arijit/badrecords/subscriptions/") \
#             .load(subs_path)

# display(subs_df); print("Total Rows: ",subs_df.count())
# subs_df = subs_df.withColumn("ingest_file_name", input_file_name()) \
#                  .withColumn("ingest_time", current_timestamp())

# display(subs_df)
# subs_df.write.format("delta").mode("overwrite").save(bronze_subs_path)
# watch_df = spark.read.format("csv") \
#             .option("header", "true") \
#             .option("inferSchema", "true") \
#             .option("badRecordsPath", "dbfs:/FileStore/tables/Arijit/badrecords/watch_history/") \
#             .load(watch_path)

# display(watch_df); print("Total Rows: ",watch_df.count())
# watch_df = watch_df.withColumn("ingest_file_name", input_file_name()) \
#                    .withColumn("ingest_time", current_timestamp())

# display(watch_df)
# watch_df.write.format("delta").mode("overwrite").save(bronze_watch_path)

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, to_date, lit, current_date

users_bronze = users_df \
                .withColumn("ingestion_time", current_timestamp()) \
                .withColumn("ingestion_date",to_date(current_timestamp())) \
                .withColumn("source_file", input_file_name())

display(users_bronze)


subs_bronze = subs_df.withColumn("ingestion_time", current_timestamp()) \
                      .withColumn("ingestion_date", current_date()) \
                      .withColumn("source_file", input_file_name())

display(subs_bronze)

watch_bronze = watch_df.withColumn("ingestion_time", current_timestamp()) \
                      .withColumn("ingestion_date", current_date()) \
                      .withColumn("source_file", input_file_name())
                    
display(watch_bronze)

payments_bronze = payments_df.withColumn("ingestion_time", current_timestamp()) \
                      .withColumn("ingestion_date", current_date()) \
                      .withColumn("source_file", input_file_name())
                    
display(payments_bronze)
# # DBTITLE 1,Create a Database
# spark.sql("DROP DATABASE IF EXISTS streaming_db CASCADE")
# spark.sql("CREATE DATABASE IF NOT EXISTS streaming_db"


# Write to Bronze (Delta)

# Spark Write — Formats and Modes (Bronze Layer Notes)

## 1. Spark Write Syntax (General)

In Spark, writing data usually follows this pattern:

```
df.write
  .format("format_name")
  .mode("write_mode")
  .option("key","value")
  .save("path")
```

Or writing to table:

```
df.write
  .format("delta")
  .mode("append")
  .saveAsTable("bronze_db.users")
```

---

# 2. Write Formats in Spark

These are the most common formats used in Data Engineering.

| Format  | Description        | Use Case                       |
| ------- | ------------------ | ------------------------------ |
| delta   | Delta Lake format  | Databricks, Bronze/Silver/Gold |
| parquet | Columnar storage   | Data lake storage              |
| csv     | Comma separated    | Raw data export/import         |
| json    | JSON files         | Semi-structured data           |
| avro    | Row-based binary   | Kafka / streaming              |
| orc     | Optimized columnar | Hive systems                   |
| text    | Plain text         | Logs                           |
| jdbc    | Write to database  | SQL Server, MySQL              |

### Example

```
df.write.format("delta").save("/mnt/bronze/users")
df.write.format("parquet").save("/mnt/bronze/users")
df.write.format("csv").option("header","true").save("/mnt/bronze/users")
```

---

# 3. Write Modes in Spark

These control what happens if data already exists.

| Mode      | Meaning                       |
| --------- | ----------------------------- |
| append    | Add new data                  |
| overwrite | Delete old data and write new |
| error     | Throw error if data exists    |
| ignore    | Do nothing if data exists     |

### Example

```
df.write.mode("append").format("delta").save(path)
df.write.mode("overwrite").format("delta").save(path)
```

---

# 4. Mode Behavior Example

Assume table already has data:

| Mode      | Result          |
| --------- | --------------- |
| append    | Old + New data  |
| overwrite | Only new data   |
| error     | Job fails       |
| ignore    | Nothing happens |

---

# 5. Most Common Usage in Medallion Architecture

| Layer  | Format | Mode              |
| ------ | ------ | ----------------- |
| Bronze | delta  | append            |
| Silver | delta  | overwrite / merge |
| Gold   | delta  | overwrite         |

### Bronze Example

```
users_df.write
  .format("delta")
  .mode("append")
  .saveAsTable("bronze_db.users")
```

---

# 6. Important Options Used While Writing

| Option          | Purpose          |
| --------------- | ---------------- |
| mergeSchema     | Update schema    |
| overwriteSchema | Replace schema   |
| partitionBy     | Partition data   |
| path            | Storage location |
| compression        | snappy/gzip                   |
| maxRecordsPerFile  | Control file size             |
| replaceWhere       | Overwrite specific partition  |
| checkpointLocation | Streaming writes              |

### Example

```
df.write
  .format("delta")
  .mode("append")
  .option("mergeSchema","true")
  .partitionBy("country")
  .save(path)
```

---

# 7. Interview Important Summary

## Formats

* delta
* parquet
* csv
* json
* avro
* orc
* jdbc
* text

## Modes

* append
* overwrite
* error
* ignore

---

# 8. One Line Summary

**Format = How data is stored**
**Mode = What to do if data already exists**


In [0]:
users_bronze.write.format("delta") \
                    .mode("append") \
                    .option("mergeSchema", "true") \
                    .partitionBy("state") \
                    .save(bronze_users_path)


subs_bronze.write.format("delta") \
                    .mode("append") \
                    .option("mergeSchema", "true") \
                    .partitionBy("platform") \
                    .save(bronze_subs_path)


watch_bronze.write.format("delta") \
                    .mode("append") \
                    .option("mergeSchema", "true") \
                    .partitionBy("platform") \
                    .option("path", bronze_watch_path) \
                    .saveAsTable("arijit_bronze_db.watch_history")


payments_bronze.write.format("delta") \
                    .mode("append") \
                    .option("mergeSchema", "true") \
                    .partitionBy("method") \
                    .option("path", bronze_payments_path) \
                    .saveAsTable("arijit_bronze_db.payments")


# # DBTITLE 1,Create a Database
# spark.sql("DROP DATABASE IF EXISTS arijit_bronze_db CASCADE")
# spark.sql("CREATE DATABASE IF NOT EXISTS arijit_bronze_db")
# spark.sql("USE arijit_bronze_db")
# spark.sql("DROP TABLE IF EXISTS users")
# spark.sql("DROP TABLE IF EXISTS subscriptions")
# spark.sql("DROP TABLE IF EXISTS watch_history")
# spark.sql("DROP TABLE IF EXISTS payments")
# spark.sql("CREATE TABLE users USING DELTA LOCATION '{}'".format(bronze_users_path))
# spark.sql("CREATE TABLE subscriptions USING DELTA LOCATION '{}'".format(bronze_subs_path))
# spark.sql("CREATE TABLE watch_history USING DELTA LOCATION '{}'".format(bronze_watch_path))
# spark.sql("CREATE TABLE payments USING DELTA LOCATION '{}'".format(bronze_payments_path))
# spark.sql("DESCRIBE DETAIL users")
# spark.sql("DESCRIBE DETAIL subscriptions")
# spark.sql("DESCRIBE DETAIL watch_history")
# spark.sql("DESCRIBE DETAIL payments")

# spark.sql("SELECT * FROM users").display()
# spark.sql("SELECT * FROM subscriptions").display()
# spark.sql("SELECT * FROM watch_history").display()
# spark.sql("SELECT * FROM payments").display()

In [0]:
%sql
SHOW TABLES IN arijit_bronze_db;

In [0]:
display(spark.read.format("delta").load(bronze_users_path))
display(spark.read.format("delta").load(bronze_subs_path))
display(spark.read.format("delta").load(bronze_watch_path))
display(spark.read.format("delta").load(bronze_payments_path))

In [0]:
bronze_users_path

In [0]:
dbutils.fs.ls('dbfs:/FileStore/tables/Arijit/bronze/users')

dbutils.fs.ls('dbfs:/FileStore/tables/Arijit/bronze/subscriptions')

# Delta Tables — Registering Tables in Hive Metastore (Short Notes)

In Databricks / Spark, there are **two main ways to register Delta tables in Hive Metastore**.

---

# Method 1 — Write First, Then Register (save + CREATE TABLE USING LOCATION)

## Step 1 — Write Delta files to storage path

```python
df.write.format("delta") \
  .mode("append") \
  .save("dbfs:/FileStore/tables/Arijit/bronze/users")
```

This creates Delta files:

```
dbfs:/FileStore/tables/Arijit/bronze/users
    |_ _delta_log
    |_ part-000.parquet
```

## Step 2 — Register table in Hive Metastore

```sql
CREATE TABLE arijit_bronze_db.users
USING DELTA
LOCATION 'dbfs:/FileStore/tables/Arijit/bronze/users';
```

### What happens:

* Data already exists in storage
* Hive Metastore stores metadata
* Table now accessible via SQL

### When to use:

* Data already exists
* Another team created Delta files
* Migrating existing Delta tables
* Registering external data

---

# Method 2 — Write and Register Together (saveAsTable with path) — Recommended

```python
df.write.format("delta") \
  .mode("append") \
  .option("path", "dbfs:/FileStore/tables/Arijit/bronze/users") \
  .saveAsTable("arijit_bronze_db.users")
```

### What happens:

This single command:

1. Writes Delta files to storage
2. Registers table in Hive Metastore

This creates an **External Table**.

### When to use:

* Bronze/Silver/Gold tables
* Data lake architecture
* Production pipelines
* Recommended best practice

---

# Difference Summary

| Method                       | Writes Data | Registers Table | Use Case               |
| ---------------------------- | ----------- | --------------- | ---------------------- |
| save(path)                   | Yes         | No              | Write only             |
| CREATE TABLE USING LOCATION  | No          | Yes             | Register existing data |
| saveAsTable()                | Yes         | Yes             | Managed table          |
| option("path") + saveAsTable | Yes         | Yes             | External table (Best)  |

---

# Managed vs External Table

| Table Type     | Data Location                  |
| -------------- | ------------------------------ |
| Managed Table  | Hive warehouse location        |
| External Table | Custom path (DBFS / ADLS / S3) |

Example Managed Table:

```python
df.write.format("delta").saveAsTable("bronze_db.users")
```

Data stored in:

```
dbfs:/user/hive/warehouse/bronze_db.db/users
```

Example External Table:

```python
df.write.format("delta") \
  .option("path", "dbfs:/FileStore/tables/Arijit/bronze/users") \
  .saveAsTable("bronze_db.users")
```

---

# Interview Important Points

* Hive Metastore stores **metadata**, not data
* Delta files stored in **DBFS / ADLS / S3**
* External tables are preferred in data lake architecture
* saveAsTable registers table in metastore
* CREATE TABLE USING LOCATION registers existing Delta path
* Managed tables store data in warehouse location
* External tables store data in custom storage path

---

# One Line Interview Answer

**There are two ways to register Delta tables in Hive Metastore: either write data to a path and register it using CREATE TABLE USING DELTA LOCATION, or write and register simultaneously using saveAsTable with a specified path, which creates an external table.**


# Create Table In Hive Metastore

In [0]:
%sql
CREATE TABLE IF NOT EXISTS arijit_bronze_db.users
USING DELTA
LOCATION 'dbfs:/FileStore/tables/Arijit/bronze/users';


CREATE TABLE IF NOT EXISTS arijit_bronze_db.subscriptions
USING DELTA
LOCATION 'dbfs:/FileStore/tables/Arijit/bronze/subscriptions';

In [0]:
%sql
SHOW TABLES IN arijit_bronze_db;

# Hive Metastore vs Path-Based Access (Delta Tables Notes)

## Without Hive Metastore (Path-Based Access)

If you don’t create a table in Hive Metastore, you must read data using the storage path every time.

Example:

```python
df = spark.read.format("delta").load("dbfs:/FileStore/tables/Arijit/bronze/users")
```

### Limitations:

* You must remember the full path
* Cannot query using SQL table name
* Harder to manage many datasets
* Not ideal for analytics or BI tools

You **cannot run SQL like this**:

```sql
SELECT * FROM bronze_db.users;
```

Because the table is not registered in Hive Metastore.

---

## With Hive Metastore (Table Access)

After registering the Delta table in Hive Metastore:

```sql
CREATE TABLE bronze_db.users
USING DELTA
LOCATION 'dbfs:/FileStore/tables/Arijit/bronze/users';
```

Now you can query using SQL:

```sql
SELECT * FROM bronze_db.users;
```

Or in PySpark:

```python
df = spark.table("bronze_db.users")
```

---

## What Hive Metastore Stores (Metadata)

Hive Metastore does **not store data**, it stores metadata:

| Metadata Stored                 |
| ------------------------------- |
| Table name                      |
| Database name                   |
| Schema (columns & data types)   |
| Table location                  |
| Partition information           |
| File format (Delta/Parquet/CSV) |

So:

* **Data stored in DBFS / ADLS / S3**
* **Metadata stored in Hive Metastore**
* **Tables point to data location**

---

## Summary

| Path-Based Table          | Metastore Table                |
| ------------------------- | ------------------------------ |
| Access using path         | Access using table name        |
| No metadata stored        | Metadata stored                |
| Cannot use SQL table name | Can use SQL                    |
| Harder to manage          | Easier for analytics           |
| Used for ingestion        | Used for analytics & reporting |

---

## Important Concept to Remember

```
Data Lake stores DATA
Hive Metastore stores METADATA
SQL queries TABLES
Tables point to DATA
```


In [0]:
df = spark.table("arijit_bronze_db.users")

display(df)